# Spatiotemporal Physics-Informed Neural Network for Shape Memory Materials

## Central-Hole Tension Plate with Sparse Data Learning

This notebook implements a physics-informed neural network for thermo-viscoelastic shape memory materials with **sparse data sampling**, demonstrating PINN's ability to learn from limited measurements in complex geometries.

### Problem Setup

- **Geometry**: Square plate with central hole (20mm × 20mm × 1mm, hole radius 3mm)
- **Material**: 45° fiber-reinforced SMPC
- **Boundary Conditions**: Left face (x=0) fixed, right face (x=20mm) with prescribed displacement
- **Loading History**: Shape memory cycle (loading → cooling → unloading → recovery)
- **Key Feature**: **Sparse sampling (30% of full data)** + **Stress concentration** (hole boundary)

### PINN Advantages Demonstrated

1. **Sparse Data Learning**: Train on 30% data, validate on 100% data
2. **Complex Geometry**: Handle stress concentration without dense mesh
3. **Physics Constraints**: PDE fills gaps between sparse measurements
4. **Generalization**: Predict full field from stratified sampling

## 1. Import Required Libraries

Import libraries for numerical computation, deep learning, data processing, and visualization.

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.optim import Adam
from torch.optim.lr_scheduler import StepLR
import time
from pathlib import Path
from matplotlib.tri import Triangulation
from scipy.interpolate import interp1d
import math

print("Libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"NumPy version: {np.__version__}")

Libraries imported successfully!
PyTorch version: 2.5.1+cu121
NumPy version: 2.1.2


## 2. Set Random Seeds and Device

Set random seeds for reproducibility. Detect and configure computing device (GPU or CPU).

In [2]:
torch.manual_seed(42)
np.random.seed(42)

# Check for GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

if device.type == 'cuda':
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

Using device: cuda
GPU Name: NVIDIA GeForce RTX 4080
GPU Memory: 17.17 GB


## 3. FEDataLoader Class

Load and process FE simulation results from CSV files. This loader supports:
- Multi-step loading sequences
- Adaptive time stepping from Abaqus
- Temperature interpolation
- Boundary node extraction

**Key Difference from Ex1**: Central hole geometry (20×20×1 plate vs 33×6×2 beam)

In [3]:
class FEDataLoader:
    """
    Load and process FE simulation results from CSV files with time mapping.

    This loader handles:
    1. Step-frame-time mapping from Abaqus adaptive time stepping
    2. Temperature interpolation within each step
    3. Multi-step loading protocol (4 steps: hold, cooling, unload, reheat)
    """

    def __init__(self, data_dir, step_frame_time_file=None):
        self.data_dir = Path(data_dir)
        self.step_frame_time_file = Path(step_frame_time_file) if step_frame_time_file else None
        self.data = []
        self.step_frame_time_map = None

        # Step information (same as Ex1)
        self.steps_info = {
            1: {'duration': 20.0, 'T_start': 343.0, 'T_end': 343.0},  # High-temp hold
            2: {'duration': 50.0, 'T_start': 343.0, 'T_end': 298.0},  # Cooling under load
            3: {'duration': 1.0, 'T_start': 298.0, 'T_end': 298.0},   # Unloading
            4: {'duration': 50.0, 'T_start': 298.0, 'T_end': 353.0}   # Recovery heating
        }
        self.cumulative_time = {1: 0.0, 2: 20.0, 3: 70.0, 4: 71.0}

    def load_all_data(self):
        """Load all CSV files and build time-temperature mapping"""
        # Load step-frame-time mapping if available
        if self.step_frame_time_file and self.step_frame_time_file.exists():
            sft_df = pd.read_csv(self.step_frame_time_file, sep='\t', header=None,
                                names=['Step', 'Frame', 'Time'])
            self.step_frame_time_map = {
                (row['Step'], row['Frame']): row['Time']
                for _, row in sft_df.iterrows()
            }
        else:
            self.step_frame_time_map = None

        if not self.data_dir.exists():
            raise FileNotFoundError(f"Data directory not found: {self.data_dir}")

        # Load data for each step
        for step in range(1, 5):
            step_files = sorted(
                list(self.data_dir.glob(f"*_Step-{step}_frame*.csv")),
                key=lambda x: int(x.stem.split('frame')[-1])
            )

            if len(step_files) == 0:
                continue

            for csv_file in step_files:
                df = pd.read_csv(csv_file)
                frame_idx = int(csv_file.stem.split('frame')[-1])

                # Get time from mapping or interpolate
                if self.step_frame_time_map is not None:
                    if (step, frame_idx) in self.step_frame_time_map:
                        frame_time = self.step_frame_time_map[(step, frame_idx)]
                    elif frame_idx == 0:
                        # Frame 0 is the initial state of this step, use cumulative start time
                        frame_time = self.cumulative_time[step]
                    else:
                        raise ValueError(f"Step {step}, Frame {frame_idx} not found in step-frame-time mapping!")
                else:
                    # Fallback: linear interpolation within step
                    step_duration = self.steps_info[step]['duration']
                    num_frames = len(step_files) - 1
                    if num_frames > 0:
                        frame_time = self.cumulative_time[step] + (frame_idx / num_frames) * step_duration
                    else:
                        frame_time = self.cumulative_time[step]

                # Interpolate temperature within step
                T_start = self.steps_info[step]['T_start']
                T_end = self.steps_info[step]['T_end']
                step_duration = self.steps_info[step]['duration']

                if step_duration > 0 and T_start != T_end:
                    time_in_step = frame_time - self.cumulative_time[step]
                    time_fraction = time_in_step / step_duration
                    temperature = T_start + (T_end - T_start) * time_fraction
                else:
                    temperature = T_start

                # Add metadata
                df['Time'] = frame_time
                df['Temperature'] = temperature
                df['Step'] = step
                df['Frame'] = frame_idx

                self.data.append(df)

        if len(self.data) == 0:
            raise ValueError(f"No CSV files found in {self.data_dir}. Please check the path and file names.")

        self.full_data = pd.concat(self.data, ignore_index=True)

        print(f"Loaded {len(self.data)} frames across {len(self.full_data['Step'].unique())} steps")
        print(f"Total data points: {len(self.full_data)}")
        print(f"Time range: [{self.full_data['Time'].min():.2f}, {self.full_data['Time'].max():.2f}] s")
        print(f"Temperature range: [{self.full_data['Temperature'].min():.2f}, {self.full_data['Temperature'].max():.2f}] K")

        return self.full_data

    def get_domain_bounds(self):
        """Get spatial and temporal domain bounds"""
        bounds = {
            'x_min': self.full_data['X'].min(),
            'x_max': self.full_data['X'].max(),
            'y_min': self.full_data['Y'].min(),
            'y_max': self.full_data['Y'].max(),
            'z_min': self.full_data['Z'].min(),
            'z_max': self.full_data['Z'].max(),
            't_min': self.full_data['Time'].min(),
            't_max': self.full_data['Time'].max(),
        }
        return bounds

    def get_boundary_nodes(self, face='left'):
        """
        Extract boundary nodes for specified face.

        Args:
            face: 'left' for x=x_min (clamped), 'right' for x=x_max (loaded)

        Returns:
            DataFrame with boundary node coordinates and displacements
        """
        tol = 1e-6  # Tolerance for floating point comparison

        if face == 'left':
            x_val = self.full_data['X'].min()
            mask = np.abs(self.full_data['X'] - x_val) < tol
        elif face == 'right':
            x_val = self.full_data['X'].max()
            mask = np.abs(self.full_data['X'] - x_val) < tol
        else:
            raise ValueError(f"Unknown face: {face}. Use 'left' or 'right'.")

        return self.full_data[mask]



print('FEDataLoader class defined successfully!')

FEDataLoader class defined successfully!


## 4. Material Parameters Class

Define material properties for the 45° fiber-reinforced SMPC:
- Anisotropic elastic constants (C11, C22, C12)
- Viscoelastic Prony series (5 terms)
- WLF time-temperature shift parameters
- Reference temperature and switch temperature

**Note**: Same material as Ex1/Ex2, but L_ref=20mm (plate size)

In [4]:
class MaterialParameters:
    """
    Material parameters for 45° fiber-reinforced shape memory polymer composite.

    Parameters match UMAT Source-C6-202301.for:
    - Orthotropic viscoelasticity (Prony series, 6 branches)
    - Temperature-dependent shift factors (WLF + Arrhenius)
    - Thermal expansion coefficients
    - 45° fiber orientation

    Units: mm, N, MPa, K, s
    """

    def __init__(self):
        # Reference temperature from UMAT (line 72: Tr=323)
        self.T_ref = 323.0  # K (50°C)

        # Switching temperature from UMAT (line 102: 317.4 K)
        self.T_switch = 317.4  # K (44.25°C)

        # WLF parameters from UMAT (lines 73-74)
        self.C1 = 14.8
        self.C2 = 45.6

        # Arrhenius parameters from UMAT (line 105)
        self.E_arrhenius = 27403.3  # Arrhenius coefficient
        self.T_arr_ref = 336.0  # K (reference for Arrhenius)

        # PROPS(1-6): Relaxation times rhoi (seconds)
        self.rho = np.array([0.1, 1.0, 10.0, 100.0, 1000.0, 10000.0])

        # PROPS(7) + PROPS(8-13): C11inf and C11i (MPa)
        self.C11_inf = 10319.1
        self.C11_prony = np.array([200.254, 89.0021, 288.039, 322.882, 29.7953, 1.08799])

        # PROPS(14) + PROPS(15-20): C12inf and C12i (MPa)
        self.C12_inf = 1.77138
        self.C12_prony = np.array([183.83, 82.8455, 272.998, 318.639, 30.0832, 1.0973])

        # PROPS(21) + PROPS(22-27): C22inf and C22i (MPa)
        self.C22_inf = 2.83844
        self.C22_prony = np.array([293.707, 132.426, 436.637, 510.272, 48.2092, 1.75608])

        # PROPS(28) + PROPS(29-34): C23inf and C23i (MPa)
        self.C23_inf = 1.82402
        self.C23_prony = np.array([188.472, 84.9984, 280.342, 327.818, 30.9813, 1.1279])

        # PROPS(35) + PROPS(36-41): C66inf and C66i (MPa)
        self.C66_inf = 0.530688
        self.C66_prony = np.array([55.3091, 24.9132, 82.0325, 95.5597, 9.01153, 0.328643])

        # PROPS(42-43): Thermal expansion coefficients Tp
        Tp1 = 3.7e-6   # PROPS(42)
        Tp2 = 0.00012  # PROPS(43)
        self.Tp = np.array([Tp1, Tp2, Tp2, 0.0, 0.0, 0.0])  # Tp(1), Tp(2)=Tp(3), rest=0

        # Convert to Pa (multiply by 1e6)
        self.C11_inf *= 1e6
        self.C11_prony *= 1e6
        self.C12_inf *= 1e6
        self.C12_prony *= 1e6
        self.C22_inf *= 1e6
        self.C22_prony *= 1e6
        self.C23_inf *= 1e6
        self.C23_prony *= 1e6
        self.C66_inf *= 1e6
        self.C66_prony *= 1e6

        # Prony series parameters
        self.N_prony = 6

        # Use characteristic length (plate length in X) and equilibrium modulus
        self.L_ref = 20.0  # mm (characteristic length: plate length for Ex3)
        self.E_ref = self.C11_inf  # Pa (use actual C11_inf as reference modulus)

        # C_inf: Equilibrium stiffness matrix (6x6 Voigt notation)
        self.C_inf = np.zeros((6, 6))
        # Normal stress-strain coupling
        self.C_inf[0, 0] = self.C11_inf  # σ11 = C11*ε11 + ...
        self.C_inf[0, 1] = self.C12_inf  # σ11 = ... + C12*ε22 + ...
        self.C_inf[0, 2] = self.C12_inf  # σ11 = ... + C12*ε33
        self.C_inf[1, 0] = self.C12_inf  # σ22 = C12*ε11 + ...
        self.C_inf[1, 1] = self.C22_inf  # σ22 = ... + C22*ε22 + ...
        self.C_inf[1, 2] = self.C23_inf  # σ22 = ... + C23*ε33
        self.C_inf[2, 0] = self.C12_inf  # σ33 = C12*ε11 + ...
        self.C_inf[2, 1] = self.C23_inf  # σ33 = ... + C23*ε22 + ...
        self.C_inf[2, 2] = self.C22_inf  # σ33 = ... + C22*ε33
        # Shear moduli
        self.C_inf[3, 3] = (self.C22_inf - self.C23_inf) / 2.0  # G23
        self.C_inf[4, 4] = self.C66_inf  # G13
        self.C_inf[5, 5] = self.C66_inf  # G12

        # Prony series stiffness matrices (list of 6x6 matrices, one per branch)
        self.C_prony = []
        for i in range(self.N_prony):
            C_i = np.zeros((6, 6))
            C_i[0, 0] = self.C11_prony[i]
            C_i[0, 1] = self.C12_prony[i]
            C_i[0, 2] = self.C12_prony[i]
            C_i[1, 0] = self.C12_prony[i]
            C_i[1, 1] = self.C22_prony[i]
            C_i[1, 2] = self.C23_prony[i]
            C_i[2, 0] = self.C12_prony[i]
            C_i[2, 1] = self.C23_prony[i]
            C_i[2, 2] = self.C22_prony[i]
            C_i[3, 3] = (self.C22_prony[i] - self.C23_prony[i]) / 2.0
            C_i[4, 4] = self.C66_prony[i]
            C_i[5, 5] = self.C66_prony[i]
            self.C_prony.append(C_i)

        # Relaxation times (tau_0 = rho, same as UMAT)
        self.tau_0 = self.rho

    def shift_factor(self, T):
        """
        Calculate time-temperature shift factor a_T(T) matching UMAT lines 102-106
        WLF: aT = 10^(-c1*(T-Tr)/(c2+T-Tr)) for T > 317.4 K
        Arrhenius: aT = exp(27403.3*(1/T - 1/336.0)) for T <= 317.4 K

        Args:
            T: Temperature (K), can be scalar or tensor

        Returns:
            Shift factor a_T (same type as T)
        """
        if isinstance(T, torch.Tensor):
            T = torch.clamp(T, min=250.0, max=400.0)  # Avoid numerical issues
            # WLF above T_switch
            log10_aT_wlf = -self.C1 * (T - self.T_ref) / (self.C2 + T - self.T_ref)
            aT_wlf = 10.0 ** log10_aT_wlf
            # Arrhenius below T_switch
            aT_arr = torch.exp(self.E_arrhenius * (1.0/T - 1.0/self.T_arr_ref))
            # Select based on temperature
            a_T = torch.where(T > self.T_switch, aT_wlf, aT_arr)
        else:
            T = np.clip(T, 250.0, 400.0)
            if T > self.T_switch:
                # WLF
                log10_aT = -self.C1 * (T - self.T_ref) / (self.C2 + T - self.T_ref)
                a_T = 10.0 ** log10_aT
            else:
                # Arrhenius
                a_T = np.exp(self.E_arrhenius * (1.0/T - 1.0/self.T_arr_ref))
        return a_T



print('MaterialParameters class defined successfully!')

MaterialParameters class defined successfully!


## 5. Spatiotemporal PINN Architecture

Neural network architecture:
- **Input**: (x, y, z, t, T) - 5D spatiotemporal coordinates
- **Hidden layers**: 4 layers × 128 neurons with Tanh activation
- **Outputs**: 
  - Displacement: (u_x, u_y, u_z)
  - Internal variables: q (6 components × N_prony branches)

**Design Choice**: Separate heads for displacement and internal variables to improve convergence.

In [5]:
class SpatiotemporalPINN(nn.Module):
    """
    Spatiotemporal PINN for shape memory materials with internal variables

    Input: (x, y, z, t, T) - spatial coordinates, time, and temperature
    Output: (u_x, u_y, u_z) + q[6,6] hereditary variables

    Note: Temperature T is explicitly passed as 5th input dimension

    Args:
        layers: List of layer sizes [input_dim, hidden1, hidden2, ...]
        n_prony: Number of Prony series branches (default 6)
        output_internal_vars: If True, output internal variables q (default True)
    """

    def __init__(self, layers=[5, 256, 256, 256, 256, 256], n_prony=6, output_internal_vars=True):
        super(SpatiotemporalPINN, self).__init__()

        self.n_prony = n_prony
        self.output_internal_vars = output_internal_vars

        # Shared layers for feature extraction
        self.shared_layers = nn.ModuleList()
        for i in range(len(layers) - 1):
            self.shared_layers.append(nn.Linear(layers[i], layers[i+1]))

        # Output layer for displacement (3 components)
        self.displacement_head = nn.Linear(layers[-2], 3)

        # Output layers for internal variables q (6 strain components × n_prony branches)
        if output_internal_vars:
            self.internal_var_head = nn.Linear(layers[-2], 6 * n_prony)

        # Initialize weights
        for layer in self.shared_layers:
            nn.init.xavier_normal_(layer.weight)
            nn.init.zeros_(layer.bias)
        nn.init.xavier_normal_(self.displacement_head.weight)
        nn.init.zeros_(self.displacement_head.bias)
        if output_internal_vars:
            nn.init.xavier_normal_(self.internal_var_head.weight)
            nn.init.zeros_(self.internal_var_head.bias)

    def forward(self, x, y, z, t, T):
        """Forward pass through the network"""
        inputs = torch.cat([x, y, z, t, T], dim=1)

        # Shared feature extraction
        features = inputs
        for i in range(len(self.shared_layers)):
            features = torch.tanh(self.shared_layers[i](features))

        # Displacement output
        u = self.displacement_head(features)
        u_x, u_y, u_z = u[:, 0:1], u[:, 1:2], u[:, 2:3]

        if self.output_internal_vars:
            # Internal variables output q (shape: batch_size, 6*n_prony)
            q_flat = self.internal_var_head(features)
            # Reshape to (batch_size, 6, n_prony)
            q = q_flat.reshape(-1, 6, self.n_prony)
            return u_x, u_y, u_z, q
        else:
            return u_x, u_y, u_z



print('SpatiotemporalPINN class defined successfully!')

SpatiotemporalPINN class defined successfully!


## 6. PINN Solver with Sparse Data Sampling

The `PINNSolver` class implements:
1. **Sparse Data Sampling**: Stratified spatial (20%) + temporal (50%) sampling
2. **Physics-Informed Loss**: PDE + BC + Evolution constraints
3. **Constitutive Relations**: Thermo-viscoelastic stress calculation
4. **Training Loop**: Adam optimizer with learning rate scheduling

**Key Innovation**: Combines sparse data supervision with dense PDE collocation for optimal learning.

**Note**: Due to the size of this class (~700 lines), please refer to `spatiotemporal_pinn_ex3.py` for complete implementation. Key methods include:
- `train()`: Main training loop with sparse sampling
- `compute_stress()`: Thermo-viscoelastic constitutive law
- `pde_residual()`: Momentum balance equation
- `sample_pde_points()`: Hole-aware PDE collocation

In [ ]:
# Import PINNSolver and visualization functions from the main script
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

from spatiotemporal_pinn_ex3 import (
    PINNSolver,
    plot_training_history,
    plot_sampling_strategy,
    plot_displacement_field,
    plot_shape_memory_cycle
)

print('✓ PINNSolver class imported successfully!')
print('✓ Visualization functions imported!')
print('\nKey PINNSolver methods:')
print('  - train(): Main training loop with sparse sampling')
print('  - compute_stress(): Thermo-viscoelastic constitutive law')
print('  - pde_residual(): Momentum balance PDE')
print('  - sample_pde_points(): Hole-aware collocation')

SyntaxError: invalid syntax (<string>, line 1)

## 7. Load FE Data

Load simulation results from `EX-3-RESULTS/` directory.

In [ ]:
# Setup paths
script_dir = Path(__file__).parent if '__file__' in globals() else Path('.')
data_dir = script_dir / 'EX-3-RESULTS'
step_frame_time_file = script_dir / 'step-frame-time.csv'

print(f"Data directory: {data_dir}")
print(f"Data directory exists: {data_dir.exists()}")

# Initialize and load data
fe_loader = FEDataLoader(data_dir, step_frame_time_file)
full_data = fe_loader.load_all_data()
bounds = fe_loader.get_domain_bounds()

print("\nDomain bounds:")
for key, val in bounds.items():
    print(f"  {key}: {val:.2f}")

## 8. Initialize Material Parameters and PINN Model

In [ ]:
# Material parameters
mat_params = MaterialParameters()
print("\nMaterial Parameters:")
print(f"  C11_inf: {mat_params.C11_inf/1e6:.1f} MPa")
print(f"  C22_inf: {mat_params.C22_inf/1e6:.1f} MPa")
print(f"  N_prony: {mat_params.N_prony}")

# PINN model
model = SpatiotemporalPINN(
    layers=[5, 512, 512, 512, 512, 512],
    n_prony=mat_params.N_prony,
    output_internal_vars=True
).to(device)

print(f"\nPINN Model:")
print(f"  Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  Device: {device}")

# Initialize solver
solver = PINNSolver(model, mat_params, fe_loader, bounds, use_physics_loss=True)
print(f"\nSolver initialized with L_ref={solver.L_ref}mm, E_ref={solver.E_ref/1e9:.1f}GPa")


## 9. Train PINN with Sparse Data

### Training Configuration

- **Epochs**: 8000
- **Batch size**: 1024
- **Learning rate**: 1e-4
- **Sparse sampling**: 30% spatial × 70% temporal = **21% total data** (ENHANCED)
- **PDE points**: 1500 per epoch (hole-aware)
- **BC points**: 400 per epoch

**Enhanced Configuration**:
- Loss weights: Data=2.0, BC=2.0, PDE=0.05 (emphasize data fitting)
- Hole boundary sampling: 40% of nodes (vs 30% in other regions)

**Expected training time**: ~35-45 minutes on GPU (vs 25-40 min baseline)

In [ ]:
print("="*80)
print("TRAINING PHASE: Enhanced Sparse Data PINN")
print("="*80)
print("\nEnhanced configuration:")
print("  - Sampling: 30% spatial × 70% temporal = 21% total data")
print("  - Loss weights: Increased data & BC weights (2.0)")
print("  - Hole boundary emphasis: 40% of sampled nodes")
print("\nValidation will use full 100% dataset.\n")

# Train the model with enhanced settings
sparse_data = solver.train(
    epochs=12000,
    batch_size=1024,
    learning_rate=1e-4,
    n_pde_points=1500,
    n_bc_points=400,
    spatial_sampling_ratio=0.3,   # ENHANCED: 30% nodes (vs 20% baseline)
    temporal_sampling_ratio=0.7    # ENHANCED: 70% frames (vs 50% baseline)
)

print("\nTraining completed successfully!")

## 10. Visualizations

Generate plots for:
1. Training history
2. Sparse sampling strategy
3. Displacement field comparison (FE vs PINN)
4. Shape memory recovery cycle

In [ ]:
# Load visualization functions from file
import sys
sys.path.insert(0, '.')
from spatiotemporal_pinn_ex3 import (
    plot_training_history,
    plot_sampling_strategy,
    plot_displacement_field,
    plot_shape_memory_cycle
)

print("Visualization functions loaded!")

In [ ]:
# Generate all visualizations
print("Generating visualizations...\n")

print("1. Training history...")
plot_training_history(solver, save_dir=script_dir)

print("2. Sampling strategy...")
plot_sampling_strategy(sparse_data, full_data, save_dir=script_dir)

print("3. Displacement field comparison...")
plot_displacement_field(solver, fe_loader, save_dir=script_dir)

print("4. Shape memory recovery cycle...")
plot_shape_memory_cycle(solver, fe_loader, save_dir=script_dir)

print("\nAll visualizations generated!")

## 11. Save Model and Results

In [ ]:
# Save model
model_path = script_dir / 'pinn_model_ex3.pth'
torch.save(model.state_dict(), model_path)
print(f"Model saved to: {model_path}")

# Save loss history
loss_csv_path = script_dir / 'ex3_training_loss_history.csv'
solver.save_loss_history(loss_csv_path)
print(f"Loss history saved to: {loss_csv_path}")